In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))

training : 803  test : 201
linear regression test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994


Linear Regression은 수량과 단가의 선형 결합으로 총액을 예측하기 때문에 두 변수의 곱셈 관계를 정확히 표현하지 못한다.
수량과 단가를 곱한 qp를 특성으로 추가하면 실제 총액 계산 공식을 표현할 수 있으므로 R^2가 1.0이 된다.

In [11]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
        round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


In [12]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s} mean {s.mean():.3f} std {s.std():.3f}")

max_depth 3    mean 0.845 std 0.019
max_depth 5    mean 0.961 std 0.003
max_depth 8    mean 0.992 std 0.000
max_depth 12   mean 0.995 std 0.000
max_depth None mean 0.995 std 0.001


max_depth = 8
depth 12와 비교했을 때 cross-validation 평균 R^2의 차이는 0.003에 불과하지만 잎의 수는 250개와 781개로 차이가 크기 때문에 성능이 비슷하면서 구조가 단순한 depth 8을 선택했다.

In [13]:
# depth 8인 Decision Tree 학습
tree8 = DecisionTreeRegressor(
    max_depth=8, random_state=0
).fit(X_train, y_train)

# 예측할 데이터
sample = pd.DataFrame({
    "quantity": [40],
    "unit_price": [200000]
})

# Linear Regression + qp에 사용할 특성 추가
sample2 = sample.assign(
    qp=sample["quantity"] * sample["unit_price"]
)

# 모델별 예측값 비교
print("linear + qp    :", int(round(lin2.predict(sample2)[0])))
print("tree (depth 8) :", int(round(tree8.predict(sample)[0])))
print("quantity × price :", 40 * 200000)

linear + qp    : 8000000
tree (depth 8) : 7362301
quantity × price : 8000000


In [14]:
import joblib

joblib.dump(lin2, "sales_model.pkl")

['sales_model.pkl']

수량 40, 단가 200,000의 실제 총액은 8,000,000원이다. Decision Tree(depth 8)는 7,362,301원으로 예측했지만, Linear Regression + qp는 실제 총액과 동일한 8,000,000원으로 예측했다. 따라서 실제 계산 규칙을 정확히 표현할 수 있는 Linear Regression + qp를 최종 모델로 선정하였다.